# 06 - Silver Reviews Data

## Objetivo

Realizar a transformação e padronização da tabela
`reviews_data`, proveniente da camada Bronze,
preparando os dados históricos de desempenho dos anúncios
para análises do mercado de Airbnb em João Pessoa - PB.

## Fonte

`airbnb_joao_pessoa.bronze.reviews`

## Granularidade

Cada registro representa um anúncio (`listing_id`) em uma
determinada data (`date`).

Chave lógica:

`listing_id + date`

## Responsabilidades

Construir a camada Silver, realizando a validação e organização dos dados de avaliações dos anúncios. Nesta etapa, serão verificadas a qualidade dos dados, possíveis valores nulos, duplicidades, consistência entre o número de avaliações e os identificadores dos avaliadores, além do intervalo de datas. A estrutura original será preservada, mantendo reviewers como uma lista, para garantir a integridade e flexibilidade dos dados para análises posteriores na camada Gold.

## Tabela de saída

`airbnb_joao_pessoa.silver.reviews_data`

In [0]:
%run ./00_config

In [0]:
%run ./00_utils

In [0]:
reviews = spark.table(BRONZE_TABLES["reviews"])

In [0]:
print(f"Quantidade de registros: {reviews.count()}")
print(f"Quantidade de colunas: {len(reviews.columns)}")

In [0]:
reviews_silver = (reviews.withColumnRenamed("date", "month_start").select("listing_id", "month_start", "num_reviews", "reviewers"))

In [0]:
display(reviews.select("num_reviews").summary())

In [0]:
# ANalisar reviewers
from pyspark.sql.functions import size

reviews_check = reviews.withColumn("reviewers_count", size("reviewers"))

display(reviews_check.select("listing_id", "date", "num_reviews", "reviewers_count"))

In [0]:
T = SILVER_TABLES["reviews"]
dq_check("silver", T, "contagem == bronze", abs(reviews_silver.count() - reviews.count()))
dq_check("silver", T, "chave única (listing_id, month_start)",
         duplicates_by_key(reviews_silver, ["listing_id", "month_start"]).count())
dq_check("silver", T, "num_reviews >= 1", reviews_silver.filter(F.col("num_reviews") < 1).count())
dq_check("silver", T, "num_reviews == tamanho de reviewers",
         reviews_silver.filter(F.col("num_reviews") != F.size("reviewers")).count())
dq_check("silver", T, "listing_id existe em silver.listings",
         reviews_silver.select("listing_id").distinct()
         .join(spark.table(SILVER_TABLES["listings"]).select("listing_id"), "listing_id", "left_anti").count())
dq_flush()

#### Escrita

In [0]:
(reviews_silver.write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLES["reviews"]))

# tabela antiga com o nome inconsistente
spark.sql(f"DROP TABLE IF EXISTS {SILVER}.reviews_data")

In [0]:
silver_reviews = spark.table("airbnb_joao_pessoa.silver.reviews")

print(f"Registros na Silver: {silver_reviews.count()}")
print(f"Colunas na Silver: {len(silver_reviews.columns)}")

display(silver_reviews.limit(10))